# Associated dimensions
We will try to model the way in which dimensions are linked.
Across all verbs, for example, how is dimension $V_i$ linked to dimension $O_i$, etc.

## 1. Graph-based approach

In [1]:
from similarity import load_eval_sentences_cached, load_og_sentences
from utils import DATA_DIR
import os
from tucker_tensor import TuckerDecomposition

dataset="fineweb-en"
random_state = 1
# we load the sample sentences only once
vector_path = os.path.join(DATA_DIR, "vectors", "fineweb_english_vectors.csv")
sentence_sample = load_eval_sentences_cached(vector_path=vector_path,
                                             dataset=dataset,
                                             seed=random_state,
                                             n_samples=10_000,
                                             )
full_dataset = load_og_sentences(vector_path=vector_path)
clean_sample = []
for vector in sentence_sample:
    cleaned = []
    for i, element in enumerate(vector):
        if element == "~":
            cleaned.append("None")
        else:
            cleaned.append(element)
    clean_sample.append(tuple(cleaned))
tucker = TuckerDecomposition.load_from_disk(
    dataset=dataset,
    method="siiSoftPlus",
    divergence="fr",
    dims=1000,
    rank=150,
    iterations=500
)


2026-01-16 15:30:02.684660: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
target_k = 7
vk_o = {}
for vso in clean_sample[:4]:
    (verb, subject, object) = vso
    # print(verb, subject, object)
    if not tucker.check_vocab(vso):
        # print("Vocabulary not found, skipping")
        continue
    v, s, o = tucker.fetch_latents(vso)
    print(v.shape)
    v_act = v[target_k]
    print(v_act)
    print(o)



(150,)
5.963547e-11
[2.54802126e-03 2.92794794e-01 4.02023224e-03 2.79805260e+01
 2.81281233e-01 7.91658554e-03 3.98109360e+01 3.41445770e+01
 8.51442397e-01 8.88013184e-01 1.32452021e-03 4.56510963e+01
 8.70884880e-02 5.46659508e+01 2.80028667e+01 4.85234718e+01
 3.90792312e-03 4.07924093e-02 3.81248561e-03 3.13350442e-03
 3.34008904e+01 5.47596436e+01 2.65272800e-04 1.76263368e-03
 9.30196233e-03 2.42500381e+01 3.95282143e-04 5.30787811e+01
 5.79284782e+01 1.09568157e-03 5.03828545e+01 3.02431049e+01
 5.59780836e-01 6.33224729e-04 3.43969911e-01 1.88321996e+00
 7.84385502e-02 2.32735038e+00 4.18573618e-01 1.85865548e-03
 6.86450209e-03 6.21445652e-04 3.31052361e+01 6.35822201e+00
 2.54598650e-04 9.09534923e-04 4.16664543e+01 2.15067412e-04
 5.02287979e+01 4.03489262e-01 3.28677483e+01 3.11828003e+01
 9.64512825e-01 9.24817801e-01 3.34387645e-02 2.53235416e+01
 3.64072495e+01 1.98638499e-01 3.99244903e-03 1.43638611e+01
 1.29823911e+00 4.52267518e-03 2.95972672e+01 4.03928207e-05
 6.2

In [3]:
k = 150
v_s = {}
v_o = {}
o_s = {}

for i in range(k):
    for j in range(k):
        v_o[(i,j)] = []

# we will build up a dictionary of tuples
for vso in clean_sample:
    (verb, subject, object) = vso
    if not tucker.check_vocab(vso):
        continue
    v,s,o = tucker.fetch_latents(vso)
    for i in range(k):
        for j in range(k):
            v_o[(i,j)].append((v[i],o[j]))


# Efficiency
Brute-forcing statistical relatedness calculations would be $O(N\cdot k^2)$.
Instead, we can vectorize / matricize and use numpy.


In [4]:
from scipy.stats import pearsonr
from tqdm import tqdm
stats = {}
count = 0
stat_counts = {i:0 for i in range(k)}
for i in tqdm(range(k)):
    for j in range(k):
        x, y = zip(*v_o[(i,j)])
        r, p = pearsonr(x, y)
        stats[(i,j)] = {"r": r, "p": p}
        if p < 0.1:
            stat_counts[i] += 1
        if p <= 0.10 and count < 1000:
            print(f"{i},{j}: r={r}, p={p}")
            count+=1

  1%|          | 1/150 [00:00<00:23,  6.26it/s]

0,2: r=0.03646096587181091, p=0.04576560645541414
0,3: r=-0.03748268261551857, p=0.040017844612747926
0,4: r=0.04631601646542549, p=0.011149310952187678
0,6: r=-0.03748275339603424, p=0.040017844612747926
0,7: r=-0.03748314455151558, p=0.040015945252207336
0,11: r=-0.03748346120119095, p=0.04001404596810447
0,13: r=-0.03751272335648537, p=0.03985856456586806
0,14: r=-0.03748329356312752, p=0.04001467905431249
0,15: r=-0.03758329525589943, p=0.03948649679120347
0,17: r=0.038244232535362244, p=0.03614294120874637
0,18: r=0.044721223413944244, p=0.014265582621451422
0,20: r=-0.037487246096134186, p=0.039993791693182595
0,21: r=-0.03751007094979286, p=0.03987307969990617
0,25: r=-0.03748907148838043, p=0.03998430049472569
0,27: r=-0.03748425468802452, p=0.0400096146024424
0,28: r=-0.03748822957277298, p=0.03998872948296975
0,30: r=-0.03748294338583946, p=0.040016578363894335
0,31: r=-0.03748265653848648, p=0.04001847774991468
0,40: r=0.03398140147328377, p=0.0626562803402429
0,42: r=-0.037

  1%|▏         | 2/150 [00:00<00:23,  6.27it/s]

1,44: r=0.16237622499465942, p=3.4771063479834043e-19
1,46: r=-0.04598274081945419, p=0.011745485178127265
1,47: r=0.10066645592451096, p=3.251750771052927e-08
1,48: r=-0.0459921695291996, p=0.011728259948581132
1,49: r=-0.044015172868967056, p=0.015874943983412396
1,50: r=-0.04602383077144623, p=0.01167042562358248
1,51: r=-0.045991480350494385, p=0.011729567407514633
1,54: r=-0.0370669886469841, p=0.04227849476559323
1,55: r=-0.04599154740571976, p=0.011729349488718873
1,56: r=-0.04599212110042572, p=0.011728259948581132
1,62: r=-0.04599252715706825, p=0.011727606267569546
1,63: r=0.10302356630563736, p=1.5366560977176545e-08
1,65: r=-0.046004071831703186, p=0.011706487980966407
1,68: r=-0.04603033885359764, p=0.011658715747722646
1,70: r=-0.04591301828622818, p=0.011873738590354751
1,73: r=-0.04130076989531517, p=0.023641020019217737
1,74: r=-0.045991525053977966, p=0.011729349488718873
1,75: r=-0.045990973711013794, p=0.011730439118593377
1,76: r=0.04671134799718857, p=0.0104771999

  2%|▏         | 3/150 [00:00<00:22,  6.44it/s]

...

 11%|█         | 16/150 [00:08<00:44,  2.98it/s]

15,130: r=0.030329711735248566, p=0.09661906993500273
15,132: r=-0.04148068279027939, p=0.02303954956054182
15,138: r=-0.030838804319500923, p=0.09114773127344765
16,3: r=0.12402546405792236, p=9.172078325646087e-12
16,4: r=-0.04437609389424324, p=0.015033282435813197
16,5: r=-0.031158803030848503, p=0.08783873668059519
16,6: r=0.12402529269456863, p=9.172499987434559e-12


 11%|█▏        | 17/150 [00:10<01:56,  1.14it/s]

16,7: r=0.1240246444940567, p=9.175030356831859e-12
16,10: r=-0.03588087484240532, p=0.04932769355828617
16,11: r=0.12402419000864029, p=9.176717649507045e-12
16,12: r=-0.03998780995607376, p=0.028457878915285832
16,13: r=0.12455865740776062, p=7.46403208726375e-12
16,14: r=0.12403672188520432, p=9.132526756105963e-12
16,15: r=0.12614893913269043, p=4.016128941926651e-12
16,18: r=-0.03812289610505104, p=0.036738155567787055
16,20: r=0.12401106208562851, p=9.223237457620976e-12
16,21: r=0.12414981424808502, p=8.742463413854311e-12
16,22: r=-0.04968510568141937, p=0.006472711479508927
16,23: r=-0.04345538839697838, p=0.01726221485661848
16,25: r=0.12400849163532257, p=9.232144726701704e-12
16,27: r=0.12403390556573868, p=9.142188806569052e-12
16,28: r=0.12402211129665375, p=9.183892033931917e-12
16,30: r=0.12402504682540894, p=9.173343367960511e-12
16,31: r=0.12402535229921341, p=9.172499987434559e-12
16,37: r=0.045005083084106445, p=0.013660310388830395
16,39: r=-0.032954130321741104, p

100%|██████████| 150/150 [01:34<00:00,  1.59it/s]


In [5]:
import numpy as np
from scipy.stats import pearsonr
from tqdm import tqdm

# r and p matrices are much faster than dicts
r_mat = np.full((k, k), np.nan, dtype=float)
p_mat = np.full((k, k), np.nan, dtype=float)
count = 0
for i in tqdm(range(k)):
    for j in range(k):
        pairs = v_o.get((i, j), None)
        if not pairs or len(pairs) < 3:
            continue  # pearsonr needs at least 2, but p-value meaningful for n>=3

        arr = np.asarray(pairs, dtype=float)   # shape (n, 2)
        x = arr[:, 0]
        y = arr[:, 1]

        r, p = pearsonr(x, y)  # same test
        r_mat[i, j] = r
        p_mat[i, j] = p

        if p <= 0.10 and count < 1000:
            print(f"{i},{j}: r={r}, p={p}")
            count+=1

# counts per i (fast, vectorized)
stat_counts = np.sum(p_mat < 0.1, axis=1)


  1%|          | 1/150 [00:00<00:28,  5.24it/s]

0,2: r=0.03646096602281806, p=0.04576578284842462
0,3: r=-0.03748267512394051, p=0.04001818118355339
0,4: r=0.04631602128348846, p=0.011149322063044902
0,6: r=-0.03748275204719966, p=0.04001777263498847
0,7: r=-0.0374831484189432, p=0.04001566751364321
0,11: r=-0.03748346179939439, p=0.04001400322363608
0,13: r=-0.03751271603453547, p=0.039858899102476544
0,14: r=-0.03748329613779574, p=0.04001488300618352
0,15: r=-0.037583302686253726, p=0.03948675120875479
0,17: r=0.03824423134888404, p=0.03614301956652526
0,18: r=0.04472122153967544, p=0.014265635275907941
0,20: r=-0.03748723968974306, p=0.03999394435301671
0,21: r=-0.037510068049005516, p=0.0398729175370464
0,25: r=-0.03748907044776288, p=0.039984226932320906
0,27: r=-0.03748425240291842, p=0.040009804774727216
0,28: r=-0.037488227891032955, p=0.03998869886297066
0,30: r=-0.03748294894293136, p=0.040016726914483956
0,31: r=-0.03748265552894743, p=0.04001828525545458
0,40: r=0.03398140624432072, p=0.06265656029126215
0,42: r=-0.0374

  1%|▏         | 2/150 [00:00<00:27,  5.37it/s]

1,13: r=-0.046084665597273596, p=0.01156020524100674
1,14: r=-0.04599377241899865, p=0.011725269128846789
1,15: r=-0.0463606573911903, p=0.011071558137174003
1,18: r=-0.03859073167224551, p=0.03448855670506719
1,20: r=-0.04600511881699298, p=0.01170455032801254
1,21: r=-0.04603845195178906, p=0.011643870839564188
1,23: r=0.07521131231144226, p=3.7074558525326565e-05
1,25: r=-0.04600482052001734, p=0.011705094609856439
1,26: r=0.09192389657100615, p=4.5317952829457246e-07
1,27: r=-0.045993999633139826, p=0.011724853911410442
1,28: r=-0.04600030405754995, p=0.011713338228843115
1,29: r=0.04010132966839506, p=0.028010377060064765
1,30: r=-0.04599146500331989, p=0.011729486502875228
1,31: r=-0.0459914664023509, p=0.011729483945392123
1,35: r=-0.040044151303470214, p=0.028234999398050353
1,40: r=0.046898756679864526, p=0.010171297809972826
1,41: r=0.041953167360319815, p=0.02152254072451968
1,42: r=-0.045994309625906137, p=0.011724287443039138
1,44: r=0.1623762141233737, p=3.477020550587803

  2%|▏         | 3/150 [00:00<00:27,  5.40it/s]

...

 10%|█         | 15/150 [00:02<00:24,  5.40it/s]

14,131: r=0.052183549320546016, p=0.004237463970720192
15,0: r=-0.03353784814426272, p=0.06616453508512943
15,4: r=-0.0366013408933397, p=0.04493709483700535
15,18: r=-0.0408696695130645, p=0.025138157359273713
15,32: r=-0.030157416977334338, p=0.09852887573416466
15,49: r=-0.039554771250390586, p=0.030221771904874503
15,66: r=-0.03008897423231836, p=0.09929587603747485
15,71: r=-0.03220626869116954, p=0.07767800926852152
15,72: r=-0.03355409022264873, p=0.06603328666255698
15,79: r=0.03160409945129207, p=0.08339541437330582
15,81: r=0.06218556866115972, p=0.0006518723421104527
15,105: r=-0.0381525938463037, p=0.036591711718546804
15,118: r=-0.04245518427094777, p=0.020006751172866953
15,130: r=0.030329715035153786, p=0.09661895169595677
15,132: r=-0.04148068572906419, p=0.02303946594171213
15,138: r=-0.030838806659937358, p=0.09114815638169464


 11%|█▏        | 17/150 [00:03<00:24,  5.39it/s]

16,3: r=0.12402545876942511, p=9.1720970314804e-12
16,4: r=-0.04437609368524686, p=0.015033376186222093
16,5: r=-0.031158804188821325, p=0.08783874386019763
16,6: r=0.12402529082677606, p=9.17269107739004e-12
16,7: r=0.12402463815874236, p=9.17500004928158e-12
16,10: r=-0.035880876984365075, p=0.04932798681600596
16,11: r=0.124024184466017, p=9.176605432959234e-12
16,12: r=-0.03998780407664148, p=0.028457872679779114
16,13: r=0.12455863950369837, p=7.464169992016385e-12
16,14: r=0.12403671877474043, p=9.132354019328788e-12
16,15: r=0.12614896152068927, p=4.016093678911547e-12
16,18: r=-0.03812289340392095, p=0.03673815050954796
16,20: r=0.12401105909986443, p=9.223168584820612e-12
16,21: r=0.12414978754816575, p=8.742503231849012e-12
16,22: r=-0.04968510337606561, p=0.006472729975892902
16,23: r=-0.043455385620487785, p=0.017262154970559697
16,25: r=0.12400848447046753, p=9.232329367219345e-12
16,27: r=0.12403389168740997, p=9.142316546992282e-12
16,28: r=0.12402211159857061, p=9.18394

100%|██████████| 150/150 [00:27<00:00,  5.44it/s]


In [6]:
import numpy as np
from scipy.stats import t
from tqdm import tqdm

r_mat = np.full((k, k), np.nan, dtype=float)
p_mat = np.full((k, k), np.nan, dtype=float)
for i in tqdm(range(k)):
    for j in range(k):
        pairs = v_o.get((i, j), None)

        a = np.asarray(pairs, dtype=float)
        x = a[:, 0]
        y = a[:, 1]
        n = x.size

        # Pearson r
        x = x - x.mean()
        y = y - y.mean()
        denom = np.sqrt((x @ x) * (y @ y))
        if denom == 0:
            continue
        r = (x @ y) / denom

        # p-value (two-sided), same as pearsonr’s core test
        df = n - 2
        # guard against tiny numerical issues when r is extremely close to ±1
        r_clip = np.clip(r, -1.0 + 1e-15, 1.0 - 1e-15)
        t_stat = r_clip * np.sqrt(df / (1.0 - r_clip * r_clip))
        p = 2.0 * t.sf(np.abs(t_stat), df)

        r_mat[i, j] = r
        p_mat[i, j] = p



stat_counts = np.sum(p_mat < 0.1, axis=1)
stat_counts

 10%|█         | 15/150 [00:02<00:23,  5.77it/s]


KeyboardInterrupt: 

We have benchmarked the following code against the previous one.
It is more efficient, and gives the same results.

# efficient version

The packing is what takes most of the time in the above: we can make this waaaay more efficient.

In [7]:
import numpy as np
from scipy.stats import t

def collect_latents(clean_sample, tucker, k, dtype=np.float32):
    """
    Build dense latent matrices V,S,O of shape (N,k) in one pass.
    """
    V_list, S_list, O_list = [], [], []

    for vso in clean_sample:
        if not tucker.check_vocab(vso):
            continue

        v, s, o = tucker.fetch_latents(vso)

        # keep first k dims
        V_list.append(np.asarray(v[:k], dtype=dtype))
        S_list.append(np.asarray(s[:k], dtype=dtype))
        O_list.append(np.asarray(o[:k], dtype=dtype))

    if not V_list:
        raise ValueError("No samples passed check_vocab / fetch_latents.")

    V = np.stack(V_list, axis=0)  # (N,k)
    S = np.stack(S_list, axis=0)
    O = np.stack(O_list, axis=0)
    return V, S, O


def corr_p_from_mats(A, B, alpha=0.10):
    """
    Compute Pearson correlation matrix r (k,k), p-values (k,k),
    and stat_counts per source dimension (k,).
    A, B: (N,k)
    """
    # Use float64 for stable sums, keep inputs possibly float32 for memory
    A64 = A.astype(np.float64, copy=False)
    B64 = B.astype(np.float64, copy=False)

    N = A64.shape[0]
    if N < 3:
        raise ValueError("Need N>=3 for meaningful p-values (df=N-2).")

    # Sufficient statistics
    sx  = A64.sum(axis=0)                 # (k,)
    sy  = B64.sum(axis=0)                 # (k,)
    sxx = np.square(A64).sum(axis=0)      # (k,)
    syy = np.square(B64).sum(axis=0)      # (k,)
    sxy = A64.T @ B64                     # (k,k)  <-- fast BLAS GEMM

    # Pearson r from sums:
    # cov = N*sxy - sx*sy
    # varx = N*sxx - sx^2
    # vary = N*syy - sy^2
    cov  = (N * sxy) - np.outer(sx, sy)   # (k,k)
    varx = (N * sxx) - (sx * sx)          # (k,)
    vary = (N * syy) - (sy * sy)          # (k,)

    denom = np.sqrt(np.outer(varx, vary)) # (k,k)

    r = np.divide(cov, denom, out=np.full_like(cov, np.nan), where=(denom != 0))

    # p-values via t-test
    df = N - 2
    r_clip = np.clip(r, -1.0 + 1e-15, 1.0 - 1e-15)
    t_stat = r_clip * np.sqrt(df / (1.0 - r_clip * r_clip))
    # p = 2.0 * t.sf(np.abs(t_stat), df)
    # one-sided positive test
    p = t.sf(t_stat, df)   # NOT abs()

    stat_counts = np.sum(p < alpha, axis=1)  # outgoing significant edges per source dim
    return r, p, stat_counts


def compute_all_directed(V, S, O, alpha=0.10):
    """
    Returns a dict with 6 directed relations:
      v_o, v_s, s_o, o_v, s_v, o_s
    Each entry contains r_mat, p_mat, stat_counts (per source dim).
    """
    results = {}

    results["v_o"] = corr_p_from_mats(V, O, alpha=alpha)
    print("v_o done")
    results["v_s"] = corr_p_from_mats(V, S, alpha=alpha)
    print("v_s done")
    results["s_o"] = corr_p_from_mats(S, O, alpha=alpha)
    print("s_o done")

    # "reverse direction" matters for counts-per-source, so compute explicitly:
    results["o_v"] = corr_p_from_mats(O, V, alpha=alpha)
    print("o_v done")
    results["s_v"] = corr_p_from_mats(S, V, alpha=alpha)
    print("s_v done")
    results["o_s"] = corr_p_from_mats(O, S, alpha=alpha)
    print("o_s done")

    return results


In [8]:
k = 150
alpha = 0.10

V, S, O = collect_latents(full_dataset, tucker, k, dtype=np.float32)
print("collected latents")

collected latents


In [9]:
results = compute_all_directed(V, S, O, alpha=alpha)

# Example: V -> O
r_vo, p_vo, counts_vo = results["v_o"]
print(counts_vo)

# Example: O -> V (different counts because direction)
r_ov, p_ov, counts_ov = results["o_v"]
print(counts_ov)

v_o done
v_s done
s_o done
o_v done
s_v done
o_s done
[35 34 59 39 40 17 42 50 35 40 47 39 61 64 61 62 55 51 57 69 41 36 36 32
 39 81 36 44 45 39 35 43 37 48 31 56 77 76 56 45 50 45 42 47 75 68 33 45
 68 74 81 51 43 43 63 40 45 46 26 48 51 50 40 46 76 35 46 37 53 40 73 67
 39 52 42 48 55 56 54 44 57 39 43 47 20 39 52 50 40 62 82 84 22 82 51 51
 72 40 46 50 63 46 47 73 39 39 61 69 49 36 61 71 54 45 32 44 51 42 44 45
 43 44 46 47 77 83 63 43 42 72 24 53 75 59 50 65 37 64 63 32 53 57 53 80
 31 45 79 44 73 61]
[49 34 51 43 59 41 43 37 66 68 72 44 40 47 34 47 56 46 61 83 28 63 60 56
 36 28 67 39 58 59 36 28 39 92 39 35 65 68 43 66 55 94 34 54 64 64 43 72
 64 48 23 36 77 38 44 20 31 42 85 29 40 55 31 56 74 66 49 50 34 72 37 77
 37 50 60 48 83 70 44 68 40 76 37 77 70 69 40 53 35 37 22 36 56 59 53 81
 65 73 34 53 75 53 28 89 31 40 51 31 43 57 56 35 60 33 45 57 44 45 45 35
 55 53 61 42 29 80 32 45 38 47 40 55 40 34 44 40 46 50 87 48 41 37 91 48
 41 45 47 54 48 44]


In [10]:
r_vo.shape

(150, 150)

In [11]:
test_tuple = ("lead", "officer", "meeting")
tucker.check_vocab(test_tuple)

True

In [12]:
v, s, o = tucker.fetch_latents(test_tuple)

In [13]:
from tucker_tensor import _role_index
top_indices = {}
roles = ["verb", "subject", "object"]
top_k = 5
for i, el in enumerate(tucker.fetch_latents(test_tuple)):
    top_indices[roles[i]]= np.argsort(el)[-top_k:][::-1]
top_indices

{'verb': array([23, 66, 41, 29, 54]),
 'subject': array([ 20, 142,  57,  32,  87]),
 'object': array([61, 95, 41, 10, 47])}

In [14]:
v_s_score = 0
v_o_score = 0
s_o_score = 0
for top_index_i in top_indices["verb"]:
    for top_index_j in top_indices["subject"]:
        v_s_score+=results["v_s"][0][top_index_i, top_index_j]
    for top_index_j in top_indices["object"]:
        v_o_score+=results["v_o"][0][top_index_i, top_index_j]
for top_index_i in top_indices["subject"]:
    for top_index_j in top_indices["object"]:
        s_o_score+=results["s_o"][0][top_index_i, top_index_j]
print(f"v_s: {v_s_score}, v_o: {v_o_score}, s_o: {s_o_score}")

v_s: 0.2147306947757266, v_o: 0.2074229915599995, s_o: 0.44881400483698497


# Does high A entail high B? $\rightarrow$ Mann-Whitney U

In [15]:
import numpy as np
from scipy.stats import mannwhitneyu

def top_quantile_enrichment(A, B, q=0.9, alpha=0.10):
    """
    A, B: (N, k)
    q: quantile threshold (e.g., 0.9 = top 10%)
    Returns:
      p_mat: (k, k) one-sided p-values
      stat_counts: (k,) number of significant outgoing edges
    """
    N, k = A.shape
    p_mat = np.ones((k, k), dtype=np.float64)

    for i in range(k):
        thresh = np.quantile(A[:, i], q)
        high_mask = A[:, i] >= thresh

        if high_mask.sum() < 2 or (~high_mask).sum() < 2:
            continue  # insufficient samples

        for j in range(k):
            b_high = B[high_mask, j]
            b_low  = B[~high_mask, j]

            # One-sided: high A => higher B
            _, p = mannwhitneyu(
                b_high,
                b_low,
                alternative="greater"
            )
            p_mat[i, j] = p

    stat_counts = np.sum(p_mat < alpha, axis=1)
    return p_mat, stat_counts

def compute_MWU(V, S, O, alpha=0.10):

    results = {}

    results["v_o"] = top_quantile_enrichment(V, O, alpha=alpha)
    print("v_o done")
    results["v_s"] = top_quantile_enrichment(V, S, alpha=alpha)
    print("v_s done")
    results["s_o"] = top_quantile_enrichment(S, O, alpha=alpha)
    print("s_o done")

    return results


In [ ]:
results_mwu = compute_MWU(V, S, O, alpha=alpha)

In [40]:
def check(test_tuple, results_mat, tensor, top_k=5):
    top_indices = {}
    roles = ["verb", "subject", "object"]
    for i, el in enumerate(tensor.fetch_latents(test_tuple)):
        top_indices[roles[i]] = np.argsort(el)[-top_k:][::-1]
    v_s_score = 0
    v_o_score = 0
    s_o_score = 0
    for top_index_i in top_indices["verb"]:
        for top_index_j in top_indices["subject"]:
            v_s_score += results_mat["v_s"][0][top_index_i, top_index_j]
        for top_index_j in top_indices["object"]:
            v_o_score += results_mat["v_o"][0][top_index_i, top_index_j]
    for top_index_i in top_indices["subject"]:
        for top_index_j in top_indices["object"]:
            s_o_score += results_mat["s_o"][0][top_index_i, top_index_j]
    print(f"v_s: {v_s_score}, v_o: {v_o_score}, s_o: {s_o_score}")
    return v_s_score, v_o_score, s_o_score
def full_check(test_tuple, tensor):
    print(test_tuple)
    return check(test_tuple, results, tensor)

In [41]:
test_tuple = ("lead", "soldier", "meeting")
full_check(test_tuple, tucker)

('lead', 'soldier', 'meeting')
v_s: 0.062038640847960555, v_o: -0.41304841137785037, s_o: -0.09963893792589588


(np.float64(0.062038640847960555),
 np.float64(-0.41304841137785037),
 np.float64(-0.09963893792589588))

In [20]:
from tucker_tensor import ExtendedTucker
extended = ExtendedTucker.extend_tucker(tucker, dataset=full_dataset, roles=["verb", "subject", "object"], min_count=50)

calculating reps (object): 100%|██████████| 482308/482308 [01:34<00:00, 5120.43it/s]


In [27]:
from tucker_tensor import ExtendedTucker
new_tucker = extended.integrate_extension(None)

{'verb': 369, 'subject': 2475, 'object': 2625}


In [28]:
k = 150
alpha = 0.10

V, S, O = collect_latents(full_dataset, new_tucker, k, dtype=np.float32)
print("collected latents")
results = compute_all_directed(V, S, O, alpha=alpha)

# Example: V -> O
r_vo, p_vo, counts_vo = results["v_o"]
print(counts_vo)

# Example: O -> V (different counts because direction)
r_ov, p_ov, counts_ov = results["o_v"]
print(counts_ov)

collected latents
v_o done
v_s done
s_o done
o_v done
s_v done
o_s done
[ 31  52  61  53  27  69  59  62  79  36  30  62  61  54  85  76  61  61
  61  66  55  63  56  73  38  67  57  46  33  70  40  58  79  56  33  61
  58  61  64  55  60  61  65  58  42  65  31  36  52  62  61  95  62  65
  72  61  73  64  24  52  62  65  76  38  71  57  62  44  62  27  61  79
  31  60  69  68  63 100  36  41  59  27  42  68  26  30  67  51  59  54
  96  48  54  64  65  77  62  71  61  32  67  35  67  73  64  53  67  61
  81  63  64  53  89  41  51 102  67  89  73  88  32  61  57  34  53  49
  64  64  59  61  90  46  61  64  49  61  55  43  84  46  60  46 104  97
  67  79  64  70  48  51]
[65 43 61 61 54 44 61 60 59 60 73 61 54 61 62 62 41 69 58 76 61 61 79 67
 42 61 67 61 61 69 61 61 44 79 39 41 60 56 43 76 70 81 60 45 69 59 61 66
 61 47 61 60 63 49 52 62 60 47 73 35 47 46 64 66 52 61 64 54 62 71 68 81
 48 43 60 60 76 65 36 66 41 63 68 73 55 75 61 62 34 65 68 61 75 61 61 68
 61 61 40 76 76 65 60 75 6

In [37]:
test_tuples = [
  ("lead", "general", "meeting"),
  ("control", "pilot", "party"),
  ("treat", "doctor", "wood"),
  ("charge", "prosecutor", "battery"),
  ("file", "clerk", "nail"),
  ("serve", "waiter", "sentence"),
  ("deliver", "courier", "speech"),
  ("compose", "musician", "email"),
  ("conduct", "scientist", "orchestra"),
  ("administer", "nurse", "exam"),
  ("pilot", "captain", "program"),
  ("inspect", "detective", "shipment"),
  ("harvest", "farmer", "data"),
  ("cultivate", "gardener", "relationship"),
  ("navigate", "sailor", "menu"),
  ("host", "comedian", "server"),
  ("model", "engineer", "dress"),
  ("print", "publisher", "circuit"),
  ("forge", "blacksmith", "document"),
  ("raise", "parent", "capital"),
  ("train", "coach", "algorithm"),
  ("grade", "teacher", "slope"),
  ("seal", "plumber", "deal"),
  ("fuel", "driver", "debate"),
  ("frame", "carpenter", "question"),
  ("cook", "chef", "books"),
  ("balance", "accountant", "bicycle"),
  ("spin", "DJ", "yarn"),
  ("track", "hunter", "package"),
  ("mine", "geologist", "insight"),
  ("bottle", "winemaker", "emotion"),
  ("plant", "farmer", "idea"),
  ("cast", "director", "shadow"),
  ("route", "dispatcher", "complaint"),
  ("edit", "journalist", "gene"),
  ("vaccinate", "doctor", "computer"),
  ("brew", "barista", "storm"),
  ("roll", "baker", "dice"),
  ("anchor", "journalist", "boat"),
  ("draft", "lawyer", "breeze"),
  ("pitch", "salesperson", "tent"),
  ("steer", "driver", "conversation"),
  ("freeze", "chef", "account"),
  ("wire", "electrician", "money"),
  ("stamp", "collector", "passport"),
  ("stream", "gamer", "water"),
  ("filter", "chemist", "photo"),
  ("diagnose", "mechanic", "disease"),
  ("monitor", "lifeguard", "network"),
  ("compile", "librarian", "code"),
]

for test_tuple in test_tuples:
    if tucker.check_vocab(test_tuple):
        full_check(test_tuple, tucker)

('charge', 'prosecutor', 'battery')
v_s: 0.13408791820015906, v_o: -0.27194031395052787, s_o: -0.42824323565169853
('train', 'coach', 'algorithm')
v_s: 0.21888052909573566, v_o: 0.15578500395087452, s_o: 0.02103650732698631
('fuel', 'driver', 'debate')
v_s: -0.16205718017077633, v_o: 0.337795161962222, s_o: 0.045739221931922605
('plant', 'farmer', 'idea')
v_s: -0.06287258298703813, v_o: 0.11418495856965435, s_o: -0.025943429346251934
('stream', 'gamer', 'water')
v_s: 0.25739600827055914, v_o: 0.5116485774670452, s_o: 0.06153186525420658


# Pickup here
The results here show that the s_o is indeed the lowest scoring of all: run some stat checks?

In [42]:
v_test = []
s_test = []
o_test = []
for test_tuple in test_tuples:
    if new_tucker.check_vocab(test_tuple):
        v_s, s_s, o_s = full_check(test_tuple, new_tucker)
        v_test.append(v_s)
        s_test.append(s_s)
        o_test.append(o_s)
print("v_s:", sum(v_test)/len(v_test))
print("v_o", sum(s_test)/len(s_test))
print("s_o", sum(o_test)/len(o_test))

('lead', 'general', 'meeting')
v_s: 0.07070752754000274, v_o: -0.41304841137785037, s_o: 0.20317024939210498
('control', 'pilot', 'party')
v_s: 0.19612615078734008, v_o: 0.19790576281896827, s_o: -0.0859385389042773
('treat', 'doctor', 'wood')
v_s: 0.2873867649049484, v_o: 0.22609463353635342, s_o: 0.304602158941207
('charge', 'prosecutor', 'battery')
v_s: 0.13408791820015906, v_o: -0.27194031395052787, s_o: -0.42824323565169853
('compose', 'musician', 'email')
v_s: 0.26621197975049665, v_o: 0.36726866908567307, s_o: 0.15672475619860343
('administer', 'nurse', 'exam')
v_s: -0.06283049545294765, v_o: 0.6030745625314644, s_o: 0.14047319705742545
('inspect', 'detective', 'shipment')
v_s: 0.21377694063470015, v_o: 0.1326706101141163, s_o: -0.14536219783231544
('harvest', 'farmer', 'data')
v_s: 0.12391965745092338, v_o: 0.08125370493680424, s_o: 0.04217137306075191
('cultivate', 'gardener', 'relationship')
v_s: 0.19932060545934513, v_o: 0.5801300903389596, s_o: -0.17690322473413767
('model'

In [144]:
results["v_s"][0][0,0]

np.float64(-0.015939744128089125)

In [145]:
p = 0.05
nodes = {key:value[0] for key, value in results.items()}
for key in results.keys():
    r_mat, p_mat, _ = results[key]
    for i in range(k):
        for j in range(k):
            nodes[key][i,j] = results[key][0][i,j] if results[key][1][i,j] <= p else 0

In [151]:
# we try to do this in the other direction
# We look for high-value V_S and V_O vertices combined with low-value S_O vertices
from tqdm import tqdm
top_picks = []
threshold = 0.06

for i in tqdm(range(k)):
    for j in range(k):
            for l in range(k):
                v_s = nodes["v_s"][i, j]
                v_o = nodes["v_o"][i, l]
                s_o = nodes["s_o"][j, l]
                if v_s > threshold and v_o > threshold:
                    if s_o == 0:
                        top_picks.append((i, j, l))
len(top_picks)

100%|██████████| 150/150 [00:01<00:00, 115.21it/s]


27

In [154]:
top_words = {el:set() for el in roles}
g = 2
for tup in top_picks:
    for i, el in enumerate(tup):
        role = roles[i]
        top_words[role].add(el)
top_preds = {el:{} for el in roles}
for role, top_inds in top_words.items():
    for ind in top_inds:
        top_preds[role][ind] = tucker.get_top_words_for_dimension(role, ind, g)[g-1][0]
top_preds

{'verb': {128: 'provide',
  37: 'laugh',
  102: 'maintain',
  72: 'enable',
  80: 'explore',
  83: 'play',
  148: 'increase',
  54: 'suggest',
  29: 'urge'},
 'subject': {2: 'one',
  130: 'research',
  7: 'we',
  137: 'Services',
  144: 'design',
  145: 'everybody',
  147: 'change',
  25: 'he',
  29: 'treatment',
  32: 'mother',
  36: 'establishment',
  46: 'bank',
  50: 'he',
  54: 'I',
  59: 'country',
  66: 'she',
  67: 'site',
  92: 'use',
  103: 'we',
  105: 'feature'},
 'object': {32: 'opportunity',
  34: 'blend',
  71: 'someone',
  106: 'effect',
  13: 'role',
  109: 'team',
  15: 'role',
  16: 'yourself',
  79: 'sense',
  115: 'student'}}

In [155]:
for pick in top_picks:
    candidate = []
    for i, el in enumerate(pick):
        role = roles[i]
        candidate.append(top_preds[role][el])
    print(tuple(candidate))

('urge', 'she', 'yourself')
('urge', 'she', 'student')
('laugh', 'one', 'sense')
('laugh', 'we', 'sense')
('laugh', 'he', 'sense')
('laugh', 'mother', 'sense')
('laugh', 'he', 'sense')
('laugh', 'I', 'sense')
('laugh', 'she', 'sense')
('laugh', 'we', 'sense')
('laugh', 'everybody', 'sense')
('suggest', 'change', 'role')
('suggest', 'change', 'role')
('enable', 'treatment', 'team')
('enable', 'use', 'team')
('explore', 'research', 'someone')
('play', 'country', 'effect')
('maintain', 'country', 'effect')
('provide', 'establishment', 'opportunity')
('provide', 'bank', 'opportunity')
('provide', 'site', 'opportunity')
('provide', 'Services', 'opportunity')
('provide', 'Services', 'blend')
('increase', 'feature', 'role')
('increase', 'feature', 'role')
('increase', 'design', 'role')
('increase', 'design', 'role')


# Core usage
Instead of calculating, we read the slices from the core.
the core is a $k\times k\times k$ cube in 3dimensional VSO space.
To obtain the relevant $k_a \times k_b$ matrix, we have to sum over all $k_c$ occurences

In [209]:
import numpy as np

def pairwise_from_core(G: np.ndarray):
    """
    G: core tensor of shape (k, k, k) with modes ordered as (V, S, O)
       i.e. G[v, s, o]

    Returns:
      v_s: (k, k) matrix by summing over O
      v_o: (k, k) matrix by summing over S
      s_o: (k, k) matrix by summing over V
    """
    if G.ndim != 3:
        raise ValueError(f"Expected a 3D core tensor, got shape {G.shape}")
    k1, k2, k3 = G.shape
    if not (k1 == k2 == k3):
        raise ValueError(f"Expected a k×k×k core, got {G.shape}")

    # Sum over the 3rd mode (O) -> V×S
    v_s = G.mean(axis=2)

    # Sum over the 2nd mode (S) -> V×O
    v_o = G.mean(axis=1)

    # Sum over the 1st mode (V) -> S×O
    s_o = G.mean(axis=0)

    return v_s, v_o, s_o


# Example usage

v_s, v_o, s_o = pairwise_from_core(tucker.core)
results_core = {}
nodes_core = {}

results_core["v_s"] = [v_s]
nodes_core["v_s"] = np.asarray(v_s)
results_core["v_o"] = [v_o]
nodes_core["v_o"] = np.asarray(v_o)

results_core["s_o"] = [s_o]
nodes_core["s_o"] = np.asarray(s_o)

In [211]:
for test_tuple in test_tuples:
    if tucker.check_vocab(test_tuple):
        print(test_tuple)
        check(test_tuple, results_core)

('charge', 'prosecutor', 'battery')
v_s: 44.318572998046875, v_o: 0.1624816507101059, s_o: 0.0002679242461454123
('train', 'coach', 'algorithm')
v_s: 35.914920806884766, v_o: 0.39318692684173584, s_o: 0.0887872651219368
('fuel', 'driver', 'debate')
v_s: 0.9470717906951904, v_o: 2.428978443145752, s_o: 1.6179509162902832
('plant', 'farmer', 'idea')
v_s: 1.8464175462722778, v_o: 2.8821463584899902, s_o: 1.0024009943008423
('stream', 'gamer', 'water')
v_s: 4.754809856414795, v_o: 6.1803083419799805, s_o: 3.6194026470184326


In [212]:
for el in ["v_s", "v_o", "s_o"]:
    print("max:", np.max(nodes_core[el]))
    print("min:", np.min(nodes_core[el]))

max: 49.611034
min: 1.1493158e-18
max: 74.95196
min: 1.3518533e-22
max: 37.42153
min: 2.7960127e-27


In [215]:
top_picks = []
min_threshold = 15
max_threshold = 0.1

for i in tqdm(range(k)):
    for j in range(k):
            for l in range(k):
                v_s = nodes_core["v_s"][i, j]
                v_o = nodes_core["v_o"][i, l]
                s_o = nodes_core["s_o"][j, l]
                if v_s > min_threshold and v_o > min_threshold:
                    if s_o < max_threshold:
                        top_picks.append((i, j, l))
len(top_picks)

100%|██████████| 150/150 [00:01<00:00, 115.13it/s]


19

In [217]:
top_words = {el:set() for el in roles}
g = 1
for tup in top_picks:
    for i, el in enumerate(tup):
        role = roles[i]
        top_words[role].add(el)
top_preds = {el:{} for el in roles}
for role, top_inds in top_words.items():
    for ind in top_inds:
        top_preds[role][ind] = tucker.get_top_words_for_dimension(role, ind, g)[g-1][0]
for pick in top_picks:
    candidate = []
    for i, el in enumerate(pick):
        role = roles[i]
        candidate.append(top_preds[role][el])
    print(tuple(candidate))

('include', 'ticket', 'follow')
('include', 'ticket', 'cleaning')
('enhance', 'integration', 'skill')
('explore', 'article', 'guide')
('have', 'series', 'idea')
('have', 'series', 'member')
('have', 'series', 'right')
('have', 'series', 'impact')
('have', 'business', 'idea')
('have', 'business', 'member')
('have', 'state', 'idea')
('have', 'state', 'member')
('tell', 'God', 'child')
('do', 'picture', 'trick')
('offer', 'university', 'insight')
('offer', 'university', 'experience')
('offer', 'university', 'comfort')
('offer', 'university', 'capability')
('offer', 'platform', 'comfort')


# Statistics instead of averaging


In [8]:
from scipy.stats import pearsonr
from tqdm import tqdm
stats = {}
count = 0
stat_counts = {i:0 for i in range(k)}
for i in tqdm(range(k)):
    for j in range(k):
        x = v_o[(i,j)]
        r, p = pearsonr(x)
        stats[(i,j)] = {"r": r, "p": p}
        if p < 0.1:
            stat_counts[i] += 1
        if p <= 0.10 and count < 1000:
            print(f"{i},{j}: r={r}, p={p}")
            count+=1

  0%|          | 0/150 [00:00<?, ?it/s]


TypeError: pearsonr() missing 1 required positional argument: 'y'